# RSNA Knee Abnormality Detection - 2.5D MIL Training (ConvNeXt-Tiny)

In [ ]:
import os
os.makedirs('/kaggle/working/own_src', exist_ok=True)


In [ ]:
print('=== MOUNT DIAGNOSTIC ===')
for d in sorted(os.listdir('/kaggle/input')):
    try: print(' ', d, '->', sorted(os.listdir('/kaggle/input/'+d))[:6])
    except Exception as e: print(' ', d, 'ERR', e)
comp_dir='/kaggle/input/rsna-knee-abnormality-detection'
for f in ['train.csv','train_series.csv','test.csv','test_series.csv']:
    print(' comp', f, os.path.exists(os.path.join(comp_dir,f)))
ts=os.path.join(comp_dir,'train_series')
print(' comp train_series/ exists:', os.path.isdir(ts), '| studies:', len(os.listdir(ts)) if os.path.isdir(ts) else 0)


In [ ]:
%%writefile /kaggle/working/own_src/preprocess.py
"""Turn each DICOM series into a canonical uint8 volume [n_slices, SIZE, SIZE].

Canonical means: fixed physical pixel size, centre crop to a fixed field of view,
standard in-plane orientation, standard slice order and a per-series intensity window.
The same `load_series` is used for the training cache and for inference.

Usage: python src/preprocess.py [train|test] [n_series_limit]
"""
import os
import sys
from multiprocessing import Pool

import cv2
import numpy as np
import pandas as pd
import pydicom

DATA = "/mnt/ssd/rsna_knee/data"
WORK = "/mnt/ssd/rsna_knee/work"

MM_PER_PX = 0.4
SIZE = 384  # 153.6 mm field of view
MAX_SLICES = 64

# plane -> (normal axis, slice-order sign, (col axis, sign), (row axis, sign)) in LPS coordinates
# Sagittal: slices right->left, image columns anterior->posterior, rows superior->inferior
# Coronal:  slices anterior->posterior, columns patient-right->left, rows superior->inferior
# Axial:    slices superior->inferior, columns patient-right->left, rows anterior->posterior
CANON = {
    0: (+1, (1, +1), (2, -1)),
    1: (+1, (0, +1), (2, -1)),
    2: (-1, (0, +1), (1, +1)),
}
PLANES = ["Sagittal", "Coronal", "Axial"]


def _read(path):
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array.astype(np.float32)
    if ds.get("PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    slope, inter = ds.get("RescaleSlope"), ds.get("RescaleIntercept")
    if slope is not None and inter is not None:
        arr = arr * float(slope) + float(inter)
    iop = [float(v) for v in ds.ImageOrientationPatient] if "ImageOrientationPatient" in ds else None
    ipp = [float(v) for v in ds.ImagePositionPatient] if "ImagePositionPatient" in ds else None
    ps = [float(v) for v in ds.PixelSpacing] if "PixelSpacing" in ds else None
    inst = float(ds.get("InstanceNumber") or 0)
    frames = list(arr) if arr.ndim == 3 else [arr]
    return [(f, iop, ipp, ps, inst + i * 1e-3) for i, f in enumerate(frames)]


def _canon_slice(img, iop, ps, plane):
    """Reorient one slice to the canonical in-plane orientation; returns (img, (row_mm, col_mm))."""
    ps = ps or [MM_PER_PX * img.shape[0] / SIZE] * 2  # no spacing: assume the image spans the crop
    if iop is None:
        return img, ps
    r, c = np.array(iop[:3]), np.array(iop[3:])
    _, (cax, csgn), (rax, rsgn) = CANON[plane]
    if abs(r[cax]) < abs(r[rax]):  # rows and columns are swapped relative to canonical
        img, r, c, ps = img.T, c, r, ps[::-1]
    if r[cax] * csgn < 0:
        img = img[:, ::-1]
    if c[rax] * rsgn < 0:
        img = img[::-1]
    return img, ps


def _resample(img, ps):
    """Centre crop to SIZE*MM_PER_PX mm, resample to MM_PER_PX, zero-pad to SIZE x SIZE."""
    out = np.zeros((SIZE, SIZE), np.float32)
    h, w = img.shape
    ch = min(h, int(round(SIZE * MM_PER_PX / ps[0])))
    cw = min(w, int(round(SIZE * MM_PER_PX / ps[1])))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])
    nh = max(1, min(SIZE, int(round(ch * ps[0] / MM_PER_PX))))
    nw = max(1, min(SIZE, int(round(cw * ps[1] / MM_PER_PX))))
    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR
    res = cv2.resize(crop, (nw, nh), interpolation=interp)
    oy, ox = (SIZE - nh) // 2, (SIZE - nw) // 2
    out[oy:oy + nh, ox:ox + nw] = res
    return out


def load_series(sdir, plane_name=None):
    """Returns (uint8 volume [n, SIZE, SIZE], info dict). Unreadable slices are skipped."""
    slices, bad = [], 0
    for fn in os.listdir(sdir):
        try:
            slices.extend(_read(os.path.join(sdir, fn)))
        except Exception:
            bad += 1
    info = {"n_raw": len(slices), "n_bad": bad}
    if not slices:
        return np.zeros((1, SIZE, SIZE), np.uint8), info

    iop = next((s[1] for s in slices if s[1] is not None), None)
    if iop is not None:
        normal = np.cross(iop[:3], iop[3:])
        plane = int(np.abs(normal).argmax())
    else:
        normal, plane = None, PLANES.index(plane_name) if plane_name in PLANES else 0
    info["plane"] = PLANES[plane]

    if normal is not None and all(s[2] is not None for s in slices):
        sgn = CANON[plane][0] * np.sign(normal[plane])
        slices.sort(key=lambda s: float(np.dot(s[2], normal)) * sgn)
        pos = [float(np.dot(s[2], normal)) for s in slices]
        info["slice_gap"] = float(np.median(np.abs(np.diff(pos)))) if len(pos) > 1 else 0.0
    else:
        slices.sort(key=lambda s: s[4])
    if len(slices) > MAX_SLICES:
        keep = np.linspace(0, len(slices) - 1, MAX_SLICES).round().astype(int)
        slices = [slices[i] for i in keep]

    vol = np.stack([_resample(*_canon_slice(s[0], s[1], s[3], plane)) for s in slices])
    lo, hi = np.percentile(vol[:, ::4, ::4], [1, 99.5])
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)
    info["n"] = len(vol)
    return (vol * 255).round().astype(np.uint8), info


def _job(args):
    split, study, series, plane, out_dir = args
    out = f"{out_dir}/{series}.npy"
    info = {"study": study, "series": series}
    if os.path.exists(out):
        return None
    try:
        vol, extra = load_series(f"{DATA}/{split}_series/{study}/{series}", plane)
        np.save(out + ".tmp.npy", vol)
        os.replace(out + ".tmp.npy", out)
        info.update(extra)
    except Exception as e:
        info["error"] = f"{type(e).__name__}: {e}"[:200]
    return info


def main():
    split = sys.argv[1] if len(sys.argv) > 1 else "train"
    limit = int(sys.argv[2]) if len(sys.argv) > 2 else None
    out_dir = f"{WORK}/cache{SIZE}_{split}"
    os.makedirs(out_dir, exist_ok=True)
    se = pd.read_csv(f"{DATA}/{split}_series.csv")
    if limit:
        se = se.sample(limit, random_state=0)
    tasks = [(split, a, b, p, out_dir) for a, b, p in zip(se.StudyInstanceUID, se.SeriesInstanceUID, se.Anatomical_Plane)]
    rows = []
    with Pool(18) as p:
        for i, r in enumerate(p.imap_unordered(_job, tasks, chunksize=4)):
            if r:
                rows.append(r)
            if i % 1000 == 0:
                print(f"{i}/{len(tasks)}", flush=True)
    df = pd.DataFrame(rows)
    meta = f"{WORK}/cache{SIZE}_{split}_meta.csv"
    if os.path.exists(meta) and len(df):
        df = pd.concat([pd.read_csv(meta), df]).drop_duplicates("series", keep="last")
    if len(df):
        df.to_csv(meta, index=False)
        print("done", len(df), "errors:", int(df["error"].notna().sum()) if "error" in df else 0)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/own_src/knee.py
"""Shared dataset and model code for training and inference."""
import math

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
PLANES = ["Sagittal", "Coronal", "Axial"]
# slot = plane index + 3 * (1 - fat_suppressed): fat-suppressed planes first
N_SLOTS = 6
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def slot_of(plane, fat_sup):
    return PLANES.index(plane) + 3 * (1 - int(fat_sup))


def build_study_table(series_df, n_slices):
    """series_df: StudyInstanceUID, SeriesInstanceUID, Fat_Suppression, Anatomical_Plane.
    n_slices: dict series -> cached slice count. Returns {study: [[series ids of slot 0], ...]}
    with each slot's candidates sorted by slice count (largest first)."""
    table = {}
    for st, sr, fs, pl in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID,
                              series_df.Fat_Suppression, series_df.Anatomical_Plane):
        if n_slices.get(sr, 0) < 3:
            continue
        table.setdefault(st, [[] for _ in range(N_SLOTS)])[slot_of(pl, fs)].append(sr)
    for slots in table.values():
        for cands in slots:
            cands.sort(key=lambda s: -n_slices[s])
        # Intelligent anatomical plane fallback: if fat-sat is missing, use non-fat-sat (and vice-versa)
        for p in range(3):
            fs_s, nonfs_s = p, p + 3
            if not slots[fs_s] and slots[nonfs_s]:
                slots[fs_s] = list(slots[nonfs_s])
            elif not slots[nonfs_s] and slots[fs_s]:
                slots[nonfs_s] = list(slots[fs_s])
    return table


def window_centres(n, k, train, lo=0.04, hi=0.96):
    """k slice indices (centres of 3-slice windows) spread over the series."""
    a, b = max(1.0, lo * (n - 1)), min(n - 2.0, hi * (n - 1))
    if b < a:
        a = b = (n - 1) / 2
    if train:
        edges = np.linspace(a, b, k + 1)
        c = edges[:-1] + np.random.rand(k) * (edges[1:] - edges[:-1])
    else:
        c = np.linspace(a, b, k)
    return np.clip(np.round(c).astype(int), 1, max(1, n - 2))


class StudyDataset(torch.utils.data.Dataset):
    """One item = one study: uint8 windows [N_SLOTS, k, 3, H, W], slot mask, slice positions, target."""

    def __init__(self, studies, table, cache_dir, targets=None, k=8, train=False, loader=None):
        self.studies, self.table, self.cache_dir = list(studies), table, cache_dir
        self.targets, self.k, self.train = targets, k, train
        self.loader = loader or (lambda s: np.load(f"{self.cache_dir}/{s}.npy", mmap_mode="r"))

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, i):
        st = self.studies[i]
        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])
        x, mask, pos = None, np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)
        for s, cands in enumerate(slots):
            if not cands:
                continue
            sr = cands[np.random.randint(len(cands))] if self.train else cands[0]
            vol = self.loader(sr)
            n = len(vol)
            c = window_centres(n, self.k, self.train)
            win = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])  # [k, 3, H, W]
            if x is None:
                x = np.zeros((N_SLOTS, self.k) + win.shape[1:], np.uint8)
            x[s], mask[s], pos[s] = win, True, c / max(n - 1, 1)
        if x is None:
            x = np.zeros((N_SLOTS, self.k, 3, 384, 384), np.uint8)
        y = self.targets[i] if self.targets is not None else np.zeros(len(LABELS), np.float32)
        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos), torch.from_numpy(np.asarray(y, np.float32))


def make_views(x, res, train, group, fov=0.92):
    """x: uint8 [N, 3, H, W] with N = n_series * group. Crops a centred `fov` fraction of the cached
    field of view and resizes to res; in training adds rotation/scale/shift and gain/gamma jitter
    that is shared within a series."""
    n = x.shape[0]
    ns = n // group
    dev = x.device
    x = x.float().div_(255.0)
    if train:
        ang = (torch.rand(ns, device=dev) - 0.5) * 2 * math.radians(12)
        sc = fov * (1 + (torch.rand(ns, device=dev) - 0.5) * 0.24)
        tx = (torch.rand(ns, device=dev) - 0.5) * 0.12
        ty = (torch.rand(ns, device=dev) - 0.5) * 0.12
        gamma = torch.exp((torch.rand(ns, device=dev) - 0.5) * 0.5)
        gain = 1 + (torch.rand(ns, device=dev) - 0.5) * 0.2
    else:
        ang = torch.zeros(ns, device=dev)
        sc = torch.full((ns,), fov, device=dev)
        tx = ty = torch.zeros(ns, device=dev)
        gamma = gain = None
    cos, sin = torch.cos(ang) * sc, torch.sin(ang) * sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    theta = theta.repeat_interleave(group, 0)
    grid = F.affine_grid(theta, (n, 3, res, res), align_corners=False)
    if x.shape[-1] > 1.5 * res:  # avoid aliasing when shrinking a lot
        x = F.avg_pool2d(x, 2) if x.shape[-1] >= 2 * res else F.interpolate(x, scale_factor=0.75, mode="bilinear", antialias=True)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    if train:
        g = gamma.repeat_interleave(group).view(n, 1, 1, 1)
        x = x.clamp_min(1e-4).pow(g) * gain.repeat_interleave(group).view(n, 1, 1, 1)
    mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)
    return (x - mean) / std


class KneeNet(nn.Module):
    """2D backbone on 3-slice windows -> tokens tagged with series slot and slice position ->
    small transformer over all windows of the study -> one attention pooling per finding."""

    def __init__(self, backbone="convnext_tiny", pretrained=True, dim=384, depth=2, drop=0.2, drop_path=0.1,
                 img_size=None):
        super().__init__()
        kw = {"drop_path_rate": drop_path} if drop_path else {}
        if img_size and backbone.startswith(("vit_", "eva", "beit")):  # fixed-grid transformers need the input size
            kw["img_size"] = img_size
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, **kw)
        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(self.enc.num_features, dim), nn.LayerNorm(dim))
        self.slot_emb = nn.Embedding(N_SLOTS, dim)
        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))
        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None
        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, len(LABELS)))
        self.cls_w = nn.Parameter(torch.randn(len(LABELS), dim) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(len(LABELS)))

    def head(self, feat, mask, pos):
        """feat [B, S, K, D], mask [B, S] bool, pos [B, S, K] in 0..1 -> logits [B, 12]."""
        b, s, k, _ = feat.shape
        slot = torch.arange(s, device=feat.device).view(1, s, 1).expand(b, s, k)
        h = self.proj(feat) + self.slot_emb(slot) + self.pos_emb(pos.unsqueeze(-1).to(feat.dtype))
        h = h.view(b, s * k, -1)
        pad = ~mask.unsqueeze(-1).expand(b, s, k).reshape(b, s * k)
        pad = pad & ~pad.all(1, keepdim=True)  # a study with no series at all attends to everything
        if self.tf is not None:
            h = self.tf(h, src_key_padding_mask=pad)
        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)  # [B, T, C]
        pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)
        return (pooled * self.cls_w).sum(-1) + self.cls_b

    def forward(self, x, mask, pos, res=256, train=False):
        """x uint8 [B, S, K, 3, H, W]."""
        b, s, k = x.shape[:3]
        keep = mask.view(-1)  # run the backbone only on series that exist
        xv = x.view(b * s, k, *x.shape[3:])[keep].flatten(0, 1)
        f = self.enc(make_views(xv, res, train, k).contiguous(memory_format=torch.channels_last))
        feat = f.new_zeros(b * s, k, f.shape[-1])
        feat[keep] = f.view(-1, k, f.shape[-1])
        return self.head(feat.view(b, s, k, -1), mask, pos)

In [ ]:
import glob
import json
import math
import multiprocessing as mp
import os
import sys
import time
import warnings
from multiprocessing import Pool

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader

SEED = 42
FOLD = 0
K = 8
RES = 256
EPOCHS = 3
BS_PER_GPU = 16
N_GPUS = max(1, torch.cuda.device_count())
BS = BS_PER_GPU * N_GPUS
LR = 2e-4
WD = 0.05

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

CNXT_SD = None
_w_cands = (glob.glob('/kaggle/input/*/convnext_tiny_in1k.pth') + glob.glob('/kaggle/input/*/*/convnext_tiny_in1k.pth') + glob.glob('/kaggle/input/*/*/*/convnext_tiny_in1k.pth'))
if _w_cands:
    CNXT_SD = torch.load(_w_cands[0], map_location='cpu', weights_only=True)
    print('offline convnext_tiny weights:', _w_cands[0])
else:
    warnings.warn("No offline convnext_tiny weights found under /kaggle/input! Falling back to pretrained download or random init.")

import timm

def make_kneenet():
    m = KneeNet('convnext_tiny', pretrained=CNXT_SD is None)
    if CNXT_SD is not None:
        m.enc.load_state_dict(CNXT_SD)
        print('loaded offline convnext_tiny in1k weights into enc')
    return m


In [ ]:
sys.path.insert(0, '/kaggle/working')
import own_src.knee as knee
from own_src.knee import LABELS, build_study_table
from own_src.preprocess import MAX_SLICES

comp_dir = '/kaggle/input/rsna-knee-abnormality-detection'
labels_dir = '/kaggle/input/rsna-knee-labels-v1'

def find_file(name):
    p = os.path.join(comp_dir, name)
    if os.path.exists(p):
        return p
    g = sorted(glob.glob('/kaggle/input/**/' + name, recursive=True))
    if g:
        return g[0]
    raise FileNotFoundError(f'{name} not found anywhere in /kaggle/input')
    raise FileNotFoundError(f'{name} not found anywhere in /kaggle/input')

train_csv_path = find_file('train.csv')
series_csv_path = find_file('train_series.csv')
print('train.csv:', train_csv_path, '| train_series.csv:', series_csv_path)
comp_dir = os.path.dirname(series_csv_path)
print('comp_dir (resolved):', comp_dir)
v1_targets_path = find_file('v1_targets.csv')
folds_path = find_file('folds.csv')

train_df = pd.read_csv(train_csv_path)
series_df = pd.read_csv(series_csv_path)
targets_df = pd.read_csv(v1_targets_path)
folds_df = pd.read_csv(folds_path)

assert list(targets_df.columns[1:]) == LABELS, f"Targets column mismatch: {list(targets_df.columns[1:])} != {LABELS}"

study_folds = dict(zip(folds_df.StudyInstanceUID, folds_df.fold))
all_studies = list(targets_df.StudyInstanceUID)
train_studies = [s for s in all_studies if study_folds.get(s, -1) != FOLD]
val_studies = [s for s in all_studies if study_folds.get(s, -1) == FOLD]
print(f"Total: {len(all_studies)} | Train: {len(train_studies)} | Val: {len(val_studies)}")

series_dir_base = os.path.join(comp_dir, 'train_series')
n_files = {}
for study, series in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID):
    sdir = os.path.join(series_dir_base, study, series)
    if os.path.exists(sdir):
        try:
            n_files[series] = min(len(os.listdir(sdir)), MAX_SLICES)
        except Exception:
            n_files[series] = 0
    else:
        n_files[series] = 0

study_table = build_study_table(series_df, n_files)
print(f"Study table built for {len(study_table)} studies")

In [ ]:
from own_src.preprocess import load_series

CACHE_DIR = '/kaggle/tmp/cache384'
os.makedirs(CACHE_DIR, exist_ok=True)

def _cache_job(args):
    study, series, plane = args
    out_path = os.path.join(CACHE_DIR, f"{series}.npy")
    if os.path.exists(out_path):
        return series, True, None
    sdir = os.path.join(comp_dir, 'train_series', study, series)
    try:
        vol, _ = load_series(sdir, plane)
        np.save(out_path + ".tmp.npy", vol)
        os.replace(out_path + ".tmp.npy", out_path)
        return series, False, None
    except Exception as e:
        return series, False, str(e)

tasks = [
    (st, sr, pl)
    for st, sr, pl in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID, series_df.Anatomical_Plane)
    if not os.path.exists(os.path.join(CACHE_DIR, f"{sr}.npy"))
]
print(f"Total series to cache: {len(tasks)} (already cached: {len(series_df) - len(tasks)})")

start_time = time.time()
MAX_CACHE_SECS = 4.5 * 3600
completed = 0
errors = 0

if tasks:
    with Pool(4) as pool:
        for i, (sr, skipped, err) in enumerate(pool.imap_unordered(_cache_job, tasks, chunksize=4)):
            completed += 1
            if err is not None:
                errors += 1
            if (i + 1) % 200 == 0 or (i + 1) == len(tasks):
                elapsed = time.time() - start_time
                print(f"Cached {i + 1}/{len(tasks)} series | Elapsed: {elapsed/60:.1f}m | Errors: {errors}")
            if time.time() - start_time > MAX_CACHE_SECS:
                print(f"Elapsed guard triggered (>4.5h)! Aborting cache build. Processed {completed}/{len(tasks)}.")
                pool.terminate()
                break

cached_count = len(glob.glob(os.path.join(CACHE_DIR, "*.npy")))
print(f"Active cache volume count in {CACHE_DIR}: {cached_count}")

In [ ]:
from own_src.knee import KneeNet, StudyDataset

target_map = {row[0]: row[1:].values.astype(np.float32) for _, row in targets_df.iterrows()}
train_targets = [target_map[s] for s in train_studies]
val_targets = [target_map[s] for s in val_studies]

g_seed = torch.Generator()
g_seed.manual_seed(SEED)

def _safe_loader(s):
    p = os.path.join(CACHE_DIR, f"{s}.npy")
    if os.path.exists(p):
        return np.load(p, mmap_mode="r")
    return np.zeros((1, 384, 384), dtype=np.uint8)

train_ds = StudyDataset(train_studies, study_table, CACHE_DIR, targets=train_targets, k=K, train=True, loader=_safe_loader)
val_ds = StudyDataset(val_studies, study_table, CACHE_DIR, targets=val_targets, k=K, train=False, loader=_safe_loader)

train_loader = DataLoader(
    train_ds, batch_size=BS, shuffle=True, generator=g_seed,
    num_workers=4, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=BS, shuffle=False,
    num_workers=4, pin_memory=True
)

net = KneeNet(backbone='convnext_tiny', pretrained=False, img_size=RES)
if CNXT_SD is not None:
    net.enc.load_state_dict(CNXT_SD)
    print('Loaded offline ImageNet-1k weights into enc.')
else:
    try:
        base = timm.create_model('convnext_tiny', pretrained=True, num_classes=0)
        net.enc.load_state_dict(base.state_dict())
        print('Loaded online pretrained weights into enc.')
    except Exception as e:
        warnings.warn(f'No weights loaded ({e}); training from random init')

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(net).cuda()
else:
    model = net.cuda()

criterion = nn.BCEWithLogitsLoss(reduction='none')
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scaler = GradScaler()

def compute_macro_auc(targets_arr, preds_arr):
    aucs = []
    for c in range(targets_arr.shape[1]):
        t_c = targets_arr[:, c]
        p_c = preds_arr[:, c]
        valid = ~np.isnan(t_c)
        if valid.sum() < 4:
            continue
        bin_c = (t_c[valid] > 0.5).astype(int)
        if len(np.unique(bin_c)) < 2:
            continue
        try:
            aucs.append(roc_auc_score(bin_c, p_c[valid]))
        except Exception:
            pass
    return float(np.mean(aucs)) if aucs else 0.0

best_val_auc = 0.0
best_sd = None

for epoch in range(EPOCHS):
    model.train()
    total_loss, total_steps = 0.0, 0
    for x, mask, pos, y in train_loader:
        x, mask, pos, y = x.cuda(), mask.cuda(), pos.cuda(), y.cuda()
        optimizer.zero_grad()
        with autocast():
            logits = model(x, mask, pos, res=RES, train=True)
            loss_mat = criterion(logits, y)
            valid_mask = ~torch.isnan(y)
            loss = (loss_mat * valid_mask).sum() / valid_mask.sum().clamp_min(1.0)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        total_steps += 1
    
    avg_train_loss = total_loss / max(1, total_steps)
    
    model.eval()
    val_preds_list, val_targs_list = [], []
    with torch.no_grad():
        for x, mask, pos, y in val_loader:
            x, mask, pos = x.cuda(), mask.cuda(), pos.cuda()
            with autocast():
                logits = model(x, mask, pos, res=RES, train=False)
            val_preds_list.append(torch.sigmoid(logits).cpu().numpy())
            val_targs_list.append(y.numpy())
    
    all_val_preds = np.concatenate(val_preds_list, axis=0) if val_preds_list else np.empty((0, 12))
    all_val_targs = np.concatenate(val_targs_list, axis=0) if val_targs_list else np.empty((0, 12))
    
    val_auc = compute_macro_auc(all_val_targs, all_val_preds)
    print(f"Epoch {epoch + 1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Macro AUC: {val_auc:.4f}")
    
    raw_model = model.module if hasattr(model, 'module') else model
    if val_auc >= best_val_auc or best_sd is None:
        best_val_auc = val_auc
        best_sd = {k: v.cpu() for k, v in raw_model.state_dict().items()}
        torch.save({'model': best_sd, 'args': {'backbone': 'convnext_tiny', 'res': RES}}, '/kaggle/working/best25d.pt')
        print(f"Saved new best checkpoint with Val AUC: {best_val_auc:.4f}")

In [ ]:
gold_mask = train_df[LABELS].notna().any(axis=1)
gold_df = train_df[gold_mask].copy()
gold_studies = list(gold_df.StudyInstanceUID)
print(f"Found {len(gold_studies)} gold studies.")

gold_targets = [target_map.get(s, np.full(len(LABELS), np.nan, dtype=np.float32)) for s in gold_studies]
gold_ds = StudyDataset(gold_studies, study_table, CACHE_DIR, targets=gold_targets, k=16, train=False, loader=_safe_loader)
gold_loader = DataLoader(gold_ds, batch_size=BS, shuffle=False, num_workers=4, pin_memory=True)

ckpt_data = torch.load('/kaggle/working/best25d.pt', map_location='cpu')
raw_model = model.module if hasattr(model, 'module') else model
raw_model.load_state_dict(ckpt_data['model'])
raw_model.eval()

gold_preds_list = []
with torch.no_grad():
    for x, mask, pos, _ in gold_loader:
        x, mask, pos = x.cuda(), mask.cuda(), pos.cuda()
        with autocast():
            logits = raw_model(x, mask, pos, res=RES, train=False)
        gold_preds_list.append(torch.sigmoid(logits).cpu().numpy())

gold_preds = np.concatenate(gold_preds_list, axis=0) if gold_preds_list else np.empty((0, 12))
gold_targs = np.array(gold_targets)
gold_macro_auc = compute_macro_auc(gold_targs, gold_preds)
print(f"Gold-58 Macro AUC: {gold_macro_auc:.4f} | Soft Val AUC: {best_val_auc:.4f}")

out_score = {
    'gold_macro_auc': float(gold_macro_auc),
    'soft_val_auc': float(best_val_auc)
}
with open('/kaggle/working/gold_score25d.json', 'w') as f:
    json.dump(out_score, f, indent=2)

val_sub_ds = StudyDataset(val_studies, study_table, CACHE_DIR, targets=val_targets, k=16, train=False, loader=_safe_loader)
val_sub_loader = DataLoader(val_sub_ds, batch_size=BS, shuffle=False, num_workers=4, pin_memory=True)

val_preds_list = []
with torch.no_grad():
    for x, mask, pos, _ in val_sub_loader:
        x, mask, pos = x.cuda(), mask.cuda(), pos.cuda()
        with autocast():
            logits = raw_model(x, mask, pos, res=RES, train=False)
        val_preds_list.append(torch.sigmoid(logits).cpu().numpy())

val_preds = np.concatenate(val_preds_list, axis=0) if val_preds_list else np.empty((0, 12))
val_df = pd.DataFrame(val_preds, columns=LABELS)
val_df.insert(0, 'StudyInstanceUID', val_studies)
val_df.to_csv('/kaggle/working/val_preds_fold0.csv', index=False)
print("Saved /kaggle/working/gold_score25d.json and /kaggle/working/val_preds_fold0.csv")